# Söyle ASR

Runs Tatar automatic speech recognition using the [Söyle](https://github.com/IS2AI/Soyle?tab=readme-ov-file) model, and provides utilities for batch-processing audio files and merging the resulting transcripts.

In [ ]:
# Mount Google Drive (optional — only needed if your audio/output files live there)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install "optimum-onnx[onnxruntime_gpu]"@git+https://github.com/huggingface/optimum-onnx.git
!pip install sentencepiece pyarrow regex huggingface-hub

## Code from the [Söyle](https://github.com/IS2AI/Soyle?tab=readme-ov-file) repository

Basic single-file usage example.

In [ ]:
# Import required modules
from optimum.onnxruntime import ORTModelForSpeechSeq2Seq
from transformers import pipeline, AutoTokenizer, AutoFeatureExtractor, AutoModelForSpeechSeq2Seq

# Set parameters
model_id = 'dhcppc0/soyle_onnx'
audio_file = "<path-to-audio-file-or-folder>"
# audio_file = "test_253.wav"

lang_id = "<|tt|>"

# Load the pre-trained model with GPU support (or change to "CPUExecutionProvider" if GPU is not available)
model = ORTModelForSpeechSeq2Seq.from_pretrained(model_id, provider="CPUExecutionProvider")

# Load the tokenizer and feature_extractor
tokenizer = AutoTokenizer.from_pretrained(model_id)
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)

# Create a pipeline for automatic speech recognition
pipe = pipeline("automatic-speech-recognition", model=model, tokenizer=tokenizer, feature_extractor=feature_extractor)

# Run inference (larger batch_size yields faster recognition, but may reduce quality)
output = pipe(audio_file, batch_size=4, generate_kwargs = {"language":lang_id})['text']
print(output)

## Batch processing

Adapts the example above to process every `.wav` file in a folder and save transcripts to a single output file.

In [ ]:
import os

audio_dir = "<path-to-audio-folder>"  # set this to your audio folder
wav_files = [os.path.join(audio_dir, f) for f in os.listdir(audio_dir) if f.endswith(".wav")]

print(f"Found {len(wav_files)} wav files")

In [ ]:
from optimum.onnxruntime import ORTModelForSpeechSeq2Seq
from transformers import pipeline, AutoTokenizer, AutoFeatureExtractor

model_id = "dhcppc0/soyle_onnx"
lang_id = "<|tt|>"

model = ORTModelForSpeechSeq2Seq.from_pretrained(model_id, provider="CUDAExecutionProvider")

tokenizer = AutoTokenizer.from_pretrained(model_id)
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)

pipe = pipeline("automatic-speech-recognition",
                model=model,
                tokenizer=tokenizer,
                feature_extractor=feature_extractor)

Check remaining GPU memory

In [ ]:
import torch, time

def print_gpu_usage():
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        used = (total - free) / 1024**3
        print(f"GPU used: {used:.2f} GB / {total/1024**3:.2f} GB")

output_path = "<path-to-output-txt>"

with open(output_path, "w", encoding="utf-8") as f:
    for i, wav in enumerate(wav_files, 1):
        print(f"[{i}/{len(wav_files)}] Processing: {os.path.basename(wav)}")
        print_gpu_usage()

        try:
            text = pipe(wav, batch_size=4, generate_kwargs={"language": lang_id})['text']
        except Exception as e:
            text = f"Error while processing: {e}"

        # write the result to the output file
        f.write(f"# id: {os.path.basename(wav)}\n{text}\n\n")

        print("Recognized text:", text)
        print_gpu_usage()
        print("-" * 40)

print(f"Done! All results saved to {output_path}")

## Merge transcripts

Concatenates all `.txt` transcript files in a folder into a single file, labeling each section by source filename.

In [ ]:
text_dir = "<path-to-transcripts-folder>"

txt_files = sorted([os.path.join(text_dir, f) for f in os.listdir(text_dir) if f.endswith(".txt")])

print(f"Found {len(txt_files)} files")

output_file = "<path-to-merged-output-txt>"

with open(output_file, "w", encoding="utf-8") as outfile:
    for i, fname in enumerate(txt_files, 1):
        with open(fname, "r", encoding="utf-8") as infile:
            content = infile.read().strip()

            outfile.write(f"### File {os.path.basename(fname)}\n")
            outfile.write(content + "\n\n")

print(f"Files merged into {output_file}")

## Merge transcripts by id

Same as above, but splits each file into `# id:`-delimited blocks first and sorts all blocks numerically by id across files, rather than just concatenating whole files in folder order.

In [ ]:
import re

text_dir = "<path-to-transcripts-folder>"

output_file = "<path-to-merged-by-id-output-txt>"

txt_files = sorted([os.path.join(text_dir, f) for f in os.listdir(text_dir) if f.endswith(".txt")])

entries = []

for fname in txt_files:
    with open(fname, "r", encoding="utf-8") as f:
        content = f.read().strip()

        blocks = content.split("# id:")
        for block in blocks:
            block = block.strip()
            if not block:
                continue
            # restore the '# id:' prefix
            block = "# id: " + block

            match = re.search(r"# id:\s*.*?(\d+)", block)
            if match:
                id_num = int(match.group(1))
                entries.append((id_num, block))

# sort by id
entries.sort(key=lambda x: x[0])

with open(output_file, "w", encoding="utf-8") as out:
    for _, block in entries:
        out.write(block.strip() + "\n\n")

print(f"Done! Final file saved: {output_file}")